# C7-cnn-transfer — Practice p24

**Type:** constrained coding · **Difficulty:** core · **Concepts:** tensor-shape-tracing, cnn-training

**Time budget:** 75 minutes

## Part I — committed hand trace (8 points)

Trace this stack from input shape `(2, 7, 211, 229)`:

1. `conv_a`: 7→13 channels, kernel 5, stride 2, padding 2;
2. `pool`: kernel 3, stride 2, padding 1;
3. `conv_b`: 13→23 channels, kernel `(3,5)`, stride `(1,2)`, padding `(1,0)`.

Before any layer is constructed or run, assign exact `(B,C,H,W)` tuples
`shape_after_a`, `shape_after_pool`, `shape_after_b`, and their ordered list
`shape_trace`. The marked verifier is unscored, fails closed on placeholders,
and reports agreement only—never observed shapes.

## Part II — construct and train the same stack (12 points)

Only after Part I is committed and the marked verifier reports `trace_ok`,
construct the exact three-layer stack as `features`, followed by
`AdaptiveAvgPool2d((2,2))`, `Flatten(1)`, and `Linear(23*2*2, 4)`.
Training uses the supplied manageable `(12,7,31,35)` synthetic crop, whose
channel count and exact convolution specifications match Part I. Use seed
`20260804`, the supplied generation order, integer labels, `CrossEntropyLoss`,
and Adam with `lr=0.025`; train exactly 16 full-batch steps. Assign
`crop_trace` by applying the floor formula by hand before running `features`,
then certify:

- the constructed layer outputs agree with `crop_trace` only after commit;
- logits have shape `(12,4)`;
- all optimizer-owned objects are exactly all model parameters;
- every parameter receives a gradient on the final backward;
- at least one parameter moves; and
- losses are finite with final loss at most `0.75 * initial loss`.

Combine these as the boolean `training_certificate`.

**Banned — zero points for Part I:** running any layer/model before the marked
verification cell; forward hooks; `torchinfo`; model summaries; third-party
shape calculators; or moving the committed tuples below the marked verifier.

**Banned — zero points for Part II:** pretrained weights, downloads/network,
changing the supplied stack/data/seed/draw order/batching/steps/hyperparameters,
using a shape API instead of the hand floor formula for `crop_trace`, or
reporting only loss without ownership/gradient/movement checks.

In [ ]:
shape_after_a = ...
shape_after_pool = ...
shape_after_b = ...
shape_trace = ...


### MARKED VERIFICATION CELL

Run only after committing Part I. It reports one agreement bit and never observed shapes.

In [ ]:
import torch
import torch.nn as nn

if (
    any(value is Ellipsis for value in (shape_after_a, shape_after_pool, shape_after_b))
    or shape_trace is Ellipsis
    or not isinstance(shape_trace, list)
    or len(shape_trace) != 3
):
    raise RuntimeError("commit three shape tuples and shape_trace before verification")
if any(not isinstance(shape, tuple) or len(shape) != 4 for shape in shape_trace):
    raise RuntimeError("each committed shape must be a four-entry tuple")

verification_stack = nn.Sequential(
    nn.Conv2d(7, 13, kernel_size=5, stride=2, padding=2),
    nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
    nn.Conv2d(13, 23, kernel_size=(3, 5), stride=(1, 2), padding=(1, 0)),
)
verify_x = torch.zeros(2, 7, 211, 229)
observed = []
for layer in verification_stack:
    verify_x = layer(verify_x)
    observed.append(tuple(verify_x.shape))
trace_ok = all(hand == seen for hand, seen in zip(shape_trace, observed))
print("trace_ok:", trace_ok)
del observed, verify_x, verification_stack


In [ ]:
if trace_ok is not True:
    raise RuntimeError("Part I must be committed and correct before Part II")

torch.set_default_dtype(torch.float64)
SEED = 20260804
torch.manual_seed(SEED)
generator = torch.Generator(device="cpu").manual_seed(SEED)
train_X = 0.04 * torch.randn(12, 7, 31, 35, generator=generator)
train_y = torch.arange(12, dtype=torch.long) % 4
train_X[train_y == 0, :, :, 3:8] += 0.8
train_X[train_y == 1, :, 10:15, :] += 0.8
train_X[train_y == 2, :, :, 20:25] -= 0.8
train_X[train_y == 3, :, 20:25, :] -= 0.8


In [ ]:
# Commit crop_trace from the floor formula before constructing/running features.
crop_trace = ...

# YOUR CODE HERE: exact features + adaptive pool + flatten + head
features = ...
model = ...

# YOUR CODE HERE: verify crop trace only now, then train exactly 16 steps.
constructed_trace_agrees = ...
loss_history = ...
optimizer_owns_exactly_model = ...
gradient_names = ...
moved_parameter_names = ...
training_certificate = ...
